In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install transformers==4.52.4

In [2]:
!pip install youtube-transcript-api

from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi


def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Raises ValueError if no 'v' parameter is found.
    """
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]
        if video_id:
            return video_id

    if host in ("youtube.com", "m.youtube.com", "music.youtube.com"):
        qs = parse_qs(parsed.query)
        if qs.get("v"):
            return qs["v"][0]

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "live", "v"):
            return parts[1]

    raise ValueError(f"No video id found in URL: {url}")


url = "https://youtu.be/kJf4MmouR-s?si=B4rjtyHrPL88X21i" # <-------------------------------- 'Put The Video Link Here'
video_id = extract_video_id(url)
api = YouTubeTranscriptApi()
fetched = api.fetch(video_id, languages=["ar", "en"])
text = "\n".join(snippet.text for snippet in fetched)
print(text)

You want to learn JavaScript. Maybe
you've been procrastinating for a while,
maybe you've already tried it and got
confused, or maybe you're just starting
out and you're like, "Where do I even
begin?" I understand, so pause and lean
in. I'm going to tell you exactly how to
learn JavaScript fast, and I mean
actually learn it, not just watch
tutorials and forget everything the next
[music] day, because that's what most
people do. And I know because I've been
there. I'm Olga, I'm a software engineer
with 8+ years [music] of experience in
the industry, and I also mentor people
who are learning to code from scratch or
willing to upskill their technical
skills [music] and get hired as a
software developer in Europe, the United
States, Canada, and worldwide. So, I see
every single mistake beginners [music]
make, and I'm going to help you skip all
of them. Let's get into it. So, first,
why JavaScript? Why not Python? Why not
something else? Here's the thing,
JavaScript runs on 98.9% [music]
of

In [3]:
from transformers import pipeline

summarizerr = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cuda:0


In [4]:
# Tokenize the transcript
tokens = summarizerr.tokenizer.encode(
    text,
    add_special_tokens=False
)

# BART can handle up to 1024 input tokens
chunk_size = 900

chunks = []

for i in range(0, len(tokens), chunk_size):

    chunk_tokens = tokens[i:i + chunk_size]

    chunk = summarizerr.tokenizer.decode(
        chunk_tokens,
        skip_special_tokens=True
    )

    chunks.append(chunk)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [5]:
summaries = []

for chunk in chunks:

    output22 = summarizerr(
        chunk,
        max_length=130,
        min_length=30,
        do_sample=False
    )

    summaries.append(output22[0]["summary_text"])

print(summaries)

["Olga: I'm a software engineer with 8+ years [music] of experience in the industry. I'm going to tell you exactly how to learn JavaScript fast. Olga: JavaScript runs on 98.9% of all websites.", 'Developers who use AI tools can complete tasks up to 55% faster. When you get stuck, don\'t just ask AI to fix your code. Ask it to quiz your arrays, paste your code and ask, "What could I do better here?"', "React has become the standard for production apps. Don't jump into React until your JavaScript is solid. The key is to build things you actually want to use. You're going to feel stupid.", '30 minutes a day, every day, beats a 10-hour binge once a week.Active recall, spaced repetition, project-based learning, and consistency.']


In [ ]:
final_summary = " ".join(summaries)

print(final_summary)

In [ ]:
pip install arabert

In [ ]:
from transformers import BertTokenizer, AutoModelForSeq2SeqLM, pipeline
from arabert.preprocess import ArabertPreprocessor

model_name="malmarjeh/mbert2mbert-arabic-text-summarization"
preprocessor = ArabertPreprocessor(model_name="")

tokenizer = BertTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline("text2text-generation",model=model,tokenizer=tokenizer)

In [ ]:
text = preprocessor.preprocess(text)

# Convert the transcript into tokens
tokens = tokenizer.encode(text, add_special_tokens=False)

# Keep each chunk safely below the 512-token limit
chunk_size = 400

chunks = []

for i in range(0, len(tokens), chunk_size):
    chunk_tokens = tokens[i:i + chunk_size]
    chunk = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
    chunks.append(chunk)

print("Number of chunks:", len(chunks))

# Summarize each chunk
summaries = []

for chunk in chunks:
    result = summarizer(
        chunk,
        pad_token_id=tokenizer.eos_token_id,
        num_beams=3,
        repetition_penalty=3.0,
        max_length=200,
        length_penalty=1.0,
        no_repeat_ngram_size=3
    )[0]["generated_text"]

    summaries.append(result)

# Combine the summaries
final_summary = " ".join(summaries)

print(final_summary)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from arabert.preprocess import ArabertPreprocessor

model_name="malmarjeh/bert2bert"
preprocessor = ArabertPreprocessor(model_name="")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer2 = pipeline("text2text-generation",model=model,tokenizer=tokenizer)


In [ ]:
text = preprocessor.preprocess(text)

# Convert the transcript into tokens
tokens2 = tokenizer.encode(text, add_special_tokens=False)

# Keep each chunk safely below the 512-token limit
chunk_size = 400

chunks2 = []

for i in range(0, len(tokens), chunk_size):
    chunk_tokens2 = tokens2[i:i + chunk_size]
    chunk2 = tokenizer.decode(chunk_tokens2, skip_special_tokens=True)
    chunks2.append(chunk2)

print("Number of chunks:", len(chunks2))

# Summarize each chunk
summaries2 = []

for chunk2 in chunks2:
    result2 = summarizer2(
        chunk2,
        pad_token_id=tokenizer.eos_token_id,
        num_beams=3,
        repetition_penalty=3.0,
        max_length=200,
        length_penalty=1.0,
        no_repeat_ngram_size=3
    )[0]["generated_text"]

    summaries2.append(result2)

# Combine the summaries
final_summary2 = " ".join(summaries2)

print(final_summary2)